# Quick start: your own CSV
Any CSV with a time column and numeric columns works. This example uses `data/raw/daily-min-temperatures.csv`; change the three settings below to point at your file.

In [ ]:
from pathlib import Path
import os
root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
os.chdir(root)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dynforecast.experiments import run_experiment

In [ ]:
def summary(runs):
    return pd.DataFrame([{"model": r["config"]["model"]["name"], "status": r["status"],
                          **{k: r.get("metrics", {}).get(k) for k in ("mae", "rmse", "mase")}}
                         for r in runs]).sort_values("rmse")

In [ ]:
def plot_forecast(run, target=0, window=0):
    with np.load(Path(run["config"]["output"]) / run["run_id"] / "predictions.npz") as a:
        fig, ax = plt.subplots(figsize=(10, 3))
        ax.plot(a["history_times"][window], a["history"][window, :, target], color="#9aa8ba", label="History")
        ax.plot(a["target_times"][window], a["truth"][window, :, target], color="#17263d", label="Observed future")
        ax.plot(a["target_times"][window], a["predictions"][window, :, target], color="#d1495b", label="Forecast")
        ax.set_title(run["config"]["model"]["name"]); ax.legend()

In [ ]:
# Change these for your data
CSV_PATH = "data/raw/daily-min-temperatures.csv"
TIME_COLUMN = "Date"   # None for row order
COLUMNS = ["Temp"]     # numeric columns to model; None = all numeric columns
TARGETS = [0]          # indices into COLUMNS to forecast

In [ ]:
from dynforecast.data import download_dataset
if not Path(CSV_PATH).exists() and CSV_PATH.endswith("daily-min-temperatures.csv"):
    download_dataset("weather", "data/raw")  # fetch the example file
pd.read_csv(CSV_PATH).head()

In [ ]:
dataset = {"name": "csv", "path": CSV_PATH, "timestamp": TIME_COLUMN, "columns": COLUMNS}
base = {"dataset": dataset, "targets": TARGETS, "history": 30, "horizon": 7, "stride": 7,
        "output": "results/notebook_csv"}
runs = [run_experiment({**base, "model": m}) for m in (
    {"name": "persistence"},
    {"name": "ridge"},
    {"name": "lstm", "epochs": 5, "width": 16},
)]
summary(runs)

In [ ]:
plot_forecast(min((r for r in runs if r["status"] == "completed"), key=lambda r: r["metrics"]["rmse"]))

# Available models
Swap any of these into the `model` entries above. Neural models take `epochs` and `width`.

In [ ]:
from dynforecast.models import MODEL_NAMES
print(", ".join(MODEL_NAMES))